In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd() / "data"

orders = pd.read_csv(data_dir / "Orders.csv")
returns = pd.read_csv(data_dir / "Returns.csv")
people = pd.read_csv(data_dir / "People.csv")

tables = {
    "Orders": orders,
    "Returns": returns,
    "People": people
}

In [2]:
for name, df in tables.items():
    print(f"\n--- {name} ---")
    print("Dimensions:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data types:")
    print(df.dtypes)
    print(df.head)


--- Orders ---
Dimensions: (10194, 21)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Data types:
Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID        object
Customer Name      object
Segment            object
Country/Region     object
City               object
State/Province     object
Postal Code        object
Region             object
Product ID         object
Category           object
Sub-Category       object
Product Name       object
Sales             float64
Quantity            int64
Discount          float64
Profit            float64
dtype: object
<bound method NDFrame.head of        Row ID        Order ID  Order Date  Ship Date       Ship Mode  \
0

In [3]:
#1. Checking Duplicates and missing values
for name, df in tables.items():
    print(f"\n ---{name} ---")
    print(f"\n Missing values: ")
    print(f"number of missing values: {df.isnull().sum().sum()}")
    print(f"\n---Duplicates---")
    print(f"Number of duplicate rows: {df.duplicated().sum()}")


 ---Orders ---

 Missing values: 
number of missing values: 0

---Duplicates---
Number of duplicate rows: 0

 ---Returns ---

 Missing values: 
number of missing values: 0

---Duplicates---
Number of duplicate rows: 0

 ---People ---

 Missing values: 
number of missing values: 0

---Duplicates---
Number of duplicate rows: 0


In [4]:
# 2. Check whether our identifiers behave as expected

print("Orders")
print("Unique Row IDs:", orders["Row ID"].nunique())
print("Unique Order IDs:", orders["Order ID"].nunique())
print("Unique Customer IDs:", orders["Customer ID"].nunique())
print("Unique Product IDs:", orders["Product ID"].nunique())
print("Order IDs are unique:", orders["Order ID"].is_unique)
print("Customer IDs are unique:", orders["Customer ID"].is_unique)
print("Product IDs are unique:", orders["Product ID"].is_unique)

print("\nReturns")
print("Unique returned Order IDs:", returns["Order ID"].nunique())
print("Returns Order IDs are unique:", returns["Order ID"].is_unique)

print("\nPeople")
print("Unique regions:", people["Region"].nunique())
print("People Region values:", people["Region"].unique())
print("Regions are unique:", people["Region"].is_unique)

Orders
Unique Row IDs: 10194
Unique Order IDs: 5111
Unique Customer IDs: 804
Unique Product IDs: 1862
Order IDs are unique: False
Customer IDs are unique: False
Product IDs are unique: False

Returns
Unique returned Order IDs: 296
Returns Order IDs are unique: True

People
Unique regions: 4
People Region values: ['West' 'East' 'Central' 'South']
Regions are unique: True


In [5]:
## To check if every Order Id in Return exists in Orders
returns["Order ID"].isin(orders["Order ID"]).all()

## To check if every region in orders exist in people
orders["Region"].isin(people["Region"]).all()


np.True_

In [6]:
##To check the date format
orders["Order Date"].head().tolist()

['1/3/2023', '1/4/2023', '1/4/2023', '1/4/2023', '1/5/2023']

In [7]:
## To convert the "Order Date" and "Ship Date" columns to datetime format, 
# i use the pd.to_datetime() function with the specified format. 
# This will ensure that the dates are properly recognized and can be used for further analysis.

orders["Order Date"] = pd.to_datetime(orders["Order Date"], format="%m/%d/%Y")
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"], format="%m/%d/%Y")

In [8]:
#To check if any shipping date is earlier than the order date, which would be an anomaly in the data.
(orders["Ship Date"] < orders["Order Date"]).sum()

np.int64(0)

In [9]:
#Inspecting numerical columns for suspicious values
orders[["Sales", "Quantity", "Discount", "Profit"]].describe()

,Sales,Quantity,Discount,Profit
count,10194.000000,10194.000000,10194.000000,10194.000000
mean,228.225854,3.791838,0.155385,28.673417
std,619.906839,2.228317,0.206249,232.465115
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.220000,2.000000,0.000000,1.760800
50%,53.910000,3.000000,0.200000,8.690000
75%,209.500000,5.000000,0.200000,29.297925
max,22638.480000,14.000000,0.800000,8399.976000


In [10]:

# To calculate average profit at each discount level
orders.groupby("Discount")["Profit"].agg(["mean", "sum", "count"])

,mean,sum,count
Discount,,,
0.00,66.338799,326718.5872,4925
0.10,94.791354,9099.9700,96
0.15,27.288298,1418.9915,52
0.20,24.576350,91079.9517,3706
0.30,-45.710633,-10513.4456,230
0.32,-88.560656,-2391.1377,27
0.40,-111.528378,-23086.3742,207
0.45,-226.646464,-2493.1111,11
0.50,-310.703456,-20506.4281,66


In [11]:
# To analyze discount performance
discountanalysis = orders.groupby("Discount").agg(
    total_sales = ("Sales", "sum"),
    total_profit = ("Profit", "sum"),
    average_profit = ("Profit", "mean"),
    order_count = ("Order ID", "nunique"),
    average_sales = ("Sales", "mean")
)

In [12]:
#defining profit margin as a percentage of total profit to total sales. This will help us understand how profitable each discount level is relative to the sales generated.

discountanalysis["profit_margin"] = (discountanalysis["total_profit"] / discountanalysis["total_sales"]*100)
discountanalysis




,total_sales,total_profit,average_profit,order_count,average_sales,profit_margin
Discount,,,,,,
0.00,1.105324e+06,326718.5872,66.338799,2714,224.431226,29.558632
0.10,5.495250e+04,9099.9700,94.791354,91,572.421844,16.559702
0.15,2.755852e+04,1418.9915,27.288298,51,529.971567,5.149012
0.20,7.739394e+05,91079.9517,24.576350,2440,208.834160,11.768357
0.30,1.044741e+05,-10513.4456,-45.710633,214,454.235074,-10.063211
0.32,1.449346e+04,-2391.1377,-88.560656,27,536.794770,-16.498047
0.40,1.164978e+05,-23086.3742,-111.528378,186,562.791101,-19.817012
0.45,5.484974e+03,-2493.1111,-226.646464,10,498.634000,-45.453472
0.50,5.891854e+04,-20506.4281,-310.703456,64,892.705152,-34.804712


In [13]:
category_analysis = orders.groupby('Category').agg(
        total_sales = ("Sales", "sum"),
        total_profit = ("Profit", "sum"),
        average_profit = ("Profit", "mean"),
        order_count = ("Order ID", "nunique"),
        average_sales = ("Sales", "mean"),
)
category_analysis["profit_margin"] = (category_analysis["total_profit"] / category_analysis["total_sales"]*100)

category_analysis.sort_values("total_profit", ascending= False)

,total_sales,total_profit,average_profit,order_count,average_sales,profit_margin
Category,,,,,,
Technology,839893.2790,146543.3756,78.575537,1560,450.344922,17.447857
Office Supplies,731893.3140,126023.4434,20.565183,3812,119.434288,17.218827
Furniture,754747.7613,19729.9956,8.964105,1819,342.911295,2.614118


In [14]:
subcategory_analysis = orders.groupby('Sub-Category').agg(
        total_sales = ("Sales", "sum"),
        total_profit = ("Profit", "sum"),
        average_profit = ("Profit", "mean"),
        order_count = ("Order ID", "nunique"),
        average_sales = ("Sales", "mean"),
)
subcategory_analysis["profit_margin"] = (subcategory_analysis["total_profit"] / subcategory_analysis["total_sales"]*100)

subcategory_analysis.sort_values("total_profit", ascending= False)

,total_sales,total_profit,average_profit,order_count,average_sales,profit_margin
Sub-Category,,,,,,
Copiers,150745.2900,56093.9365,801.341950,70,2153.504143,37.211071
Phones,331842.6400,45050.8265,49.890173,826,367.489081,13.575961
Accessories,167380.3180,41936.6357,54.111788,718,215.974604,25.054700
Paper,79540.5380,34511.5070,24.936060,1205,57.471487,43.388576
Binders,207354.8810,31426.1003,20.301098,1339,133.950182,15.155708
Chairs,335768.2490,27223.5323,42.939325,591,529.602916,8.107834
Storage,224644.5540,21285.1115,24.865784,787,262.435227,9.475018
Appliances,108213.1850,18329.4844,38.669798,459,228.297859,16.938310
Furnishings,95598.1260,13891.7430,13.767833,919,94.745417,14.531397
